In [2]:
import os
import sys
import gc
import time
import argparse
import numpy as np
import sentencepiece as spm
import math
import torch
import torch.nn.functional as F
import mlx.core as mx
import mlx.nn as nn
from mlx.utils import tree_flatten
import matplotlib.pyplot as plt  # 🌟 新增這行

# ==========================================
# 1. Config
# ==========================================
class Mamba3Config:
    def __init__(
        self,
        d_model=768,
        d_state=64,
        d_head=64,
        n_groups=1,
        mimo_rank=4,
        expand=4,
        num_layers=15,
        rms_norm_eps=1e-5,
        use_kmoe=True,
        kmoe_num_experts=1024,
        kmoe_top_k=2,
        vocab_size=32000
    ):
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.d_state = d_state
        self.d_head = d_head
        self.expand = expand
        self.num_layers = num_layers

        self.d_inner = int(expand * d_model)
        self.n_heads = self.d_inner // d_head
        self.n_groups = n_groups
        self.mimo_rank = mimo_rank
        self.rms_norm_eps = rms_norm_eps

        self.use_kmoe = use_kmoe
        self.kmoe_num_experts = kmoe_num_experts
        self.kmoe_top_k = kmoe_top_k

def get_factors(n):
    for i in range(int(math.sqrt(n)), 0, -1):
        if n % i == 0: return i, n // i
    return 1, n

# ==========================================
# 2. Kronecker MoE
# ==========================================
class KroneckerMoE(nn.Module):
    def __init__(self, dim_in1, dim_in2, dim_out1, dim_out2, num_experts=1024, top_k=2):
        super().__init__()
        self.dim_in1 = dim_in1
        self.dim_in2 = dim_in2
        self.dim_out1 = dim_out1
        self.dim_out2 = dim_out2
        self.num_experts = num_experts
        self.top_k = min(top_k, num_experts)

        self.router = nn.Linear(dim_in1 * dim_in2, num_experts, bias=False)
        
        std_A = (1.0 / math.sqrt(dim_in1 * dim_out1)) ** 0.5
        std_B = (1.0 / math.sqrt(dim_in2 * dim_out2)) ** 0.5
        
        self.A_experts = mx.random.normal((num_experts, dim_out1, dim_in1)) * std_A
        self.B_experts = mx.random.normal((num_experts, dim_out2, dim_in2)) * std_B

        self.scale = mx.ones((1,))
        self.bias = mx.zeros((dim_out1 * dim_out2,))
        self.out_norm = nn.RMSNorm(dim_out1 * dim_out2)
        
        self.step_confidence = mx.array(0.0) 
        self.step_indices = mx.array([0]) 

    def __call__(self, x):
        orig_shape = x.shape
        x_flat = x.reshape(-1, self.dim_in1 * self.dim_in2)
        
        # ====================================================
        # 🌟 關鍵修復：還原 PyTorch F.layer_norm 數學邏輯
        # 扣除平均值 (Zero-mean) 並除以變異數開根號 (Unit-variance)
        # ====================================================
        mean = x_flat.mean(axis=-1, keepdims=True)
        var = x_flat.var(axis=-1, keepdims=True)
        x_routed = (x_flat - mean) * mx.rsqrt(var + 1e-5)
        
        router_logits = self.router(x_routed)
        router_logits = 10.0 * mx.tanh(router_logits / 10.0) / 2.0
        
        top_k_indices = mx.argpartition(-router_logits, self.top_k - 1, axis=-1)[:, :self.top_k]
        
        global_probs = mx.softmax(router_logits, axis=-1)
        top_k_global_probs = mx.take_along_axis(global_probs, top_k_indices, axis=-1)
        self.step_confidence = top_k_global_probs.sum(axis=-1).mean()
        self.step_indices = top_k_indices 
        
        top_k_vals = mx.take_along_axis(router_logits, top_k_indices, axis=-1)
        top_k_probs = mx.softmax(top_k_vals, axis=-1)

        A_gathered = self.A_experts[top_k_indices] 
        B_gathered = self.B_experts[top_k_indices] 

        x_view = x_flat.reshape(-1, 1, self.dim_in1, self.dim_in2)
        
        M = mx.matmul(A_gathered, x_view) 
        Y_k = mx.matmul(M, B_gathered.transpose(0, 1, 3, 2)) 
        
        probs_view = top_k_probs[:, :, None, None]
        Y = (Y_k * probs_view).sum(axis=1) 
        
        Y = Y.reshape(*orig_shape[:-1], -1)
        Y = Y * self.scale + self.bias
        return self.out_norm(Y)
    

# ==========================================
# 4. K-MoE Transformer FeedForward
# ==========================================
class KMoEFeedForward(nn.Module):
    def __init__(self, config: Mamba3Config):
        super().__init__()
        d1, d2 = get_factors(config.d_model)
        f1, f2 = get_factors(config.d_model * 4)

        self.up_proj = KroneckerMoE(d1, d2, f1, f2, config.kmoe_num_experts, config.kmoe_top_k)
        self.down_proj = KroneckerMoE(f1, f2, d1, d2, config.kmoe_num_experts, config.kmoe_top_k)

    def __call__(self, x):
        return self.down_proj(nn.gelu(self.up_proj(x)))


# ==========================================
# 6. Hybrid Backbone 
# ==========================================
class LayerContainer(nn.Module):
    def __init__(self, l_type, norm, block):
        super().__init__()
        self.l_type = l_type
        if norm is not None:
            self.norm = norm
        self.block = block

# ==========================================
# 3. Mamba-3 Block
# ==========================================
class Mamba3Block(nn.Module):
    # __init__ 和 apply_rope 保持原樣不變
    def __init__(self, config: Mamba3Config):
        super().__init__()
        self.config = config
        d_in, H, G, P, N, R = config.d_model, config.n_heads, config.n_groups, config.d_head, config.d_state, config.mimo_rank
        self.ratio = H // G
        
        self.dim_z = H * P
        self.dim_x = H * P
        self.dim_B = G * N * R
        self.dim_C = G * N * R
        self.dim_dt = G
        self.dim_A = G
        self.dim_lambda = G

        d_proj_total = self.dim_z + self.dim_x + self.dim_B + self.dim_C + self.dim_dt + self.dim_A + self.dim_lambda
        self.in_proj = nn.Linear(d_in, d_proj_total, bias=True)

        if config.use_kmoe:
            p1, p2 = get_factors(P)
            q1, q2 = get_factors(P * R)
            self.x_up_proj = KroneckerMoE(p1, p2, q1, q2, config.kmoe_num_experts, config.kmoe_top_k)
            
            d_inner_f1, d_inner_f2 = get_factors(config.d_inner)
            d_in_f1, d_in_f2 = get_factors(d_in)
            self.out_proj = KroneckerMoE(d_inner_f1, d_inner_f2, d_in_f1, d_in_f2, config.kmoe_num_experts, config.kmoe_top_k)
        else:
            self.x_up_proj = nn.Linear(P, P * R, bias=False)
            self.out_proj = nn.Linear(config.d_inner, d_in, bias=False)

        self.y_down_proj = nn.Linear(P * R, P, bias=False)
        self.theta_log = mx.random.normal((G, N // 2))
        self.D = mx.ones((H,))

        self.norm_B = nn.RMSNorm(N * R, eps=config.rms_norm_eps)
        self.norm_C = nn.RMSNorm(N * R, eps=config.rms_norm_eps)
        self.bias_B = mx.ones((G, N, R))
        self.bias_C = mx.ones((G, N, R))

        self.pre_gate_norm = nn.RMSNorm(H * P)

    def apply_rope(self, x, angles):
        N_half = angles.shape[-1]
        x_reshaped = x.reshape(*x.shape[:-2], N_half, 2, x.shape[-1])
        x_transposed = x_reshaped.transpose(0, 1, 2, 3, 5, 4) 
        
        x1, x2 = x_transposed[..., 0], x_transposed[..., 1]
        cos_a = mx.cos(angles)[..., None]
        sin_a = mx.sin(angles)[..., None]
        
        rx1 = x1 * cos_a - x2 * sin_a
        rx2 = x1 * sin_a + x2 * cos_a
        
        x_rotated = mx.stack([rx1, rx2], axis=-1)
        x_rotated = x_rotated.transpose(0, 1, 2, 3, 5, 4)
        return x_rotated.reshape(x.shape)

    def __call__(self, u, cache=None):
        B_sz, L, _ = u.shape
        H, G, P, N, R = self.config.n_heads, self.config.n_groups, self.config.d_head, self.config.d_state, self.config.mimo_rank
        ratio = self.ratio

        projected = self.in_proj(u)
        
        offset = 0
        z = projected[..., offset:offset+self.dim_z]; offset += self.dim_z
        x_prime = projected[..., offset:offset+self.dim_x]; offset += self.dim_x
        B_param = projected[..., offset:offset+self.dim_B]; offset += self.dim_B
        C_param = projected[..., offset:offset+self.dim_C]; offset += self.dim_C
        dt = projected[..., offset:offset+self.dim_dt]; offset += self.dim_dt
        A_param = projected[..., offset:offset+self.dim_A]; offset += self.dim_A
        lambda_param = projected[..., offset:offset+self.dim_lambda]

        x_prime = x_prime.reshape(B_sz, L, H, P)
        
        dt = mx.logaddexp(mx.zeros_like(dt), dt) 
        A = -mx.exp(A_param)
        theta = mx.exp(self.theta_log)

        dt = (dt[:, :, :, None] * mx.ones((1, 1, 1, ratio))).reshape(B_sz, L, H)
        A_broadcast = (A[:, :, :, None] * mx.ones((1, 1, 1, ratio))).reshape(B_sz, L, H)
        theta_broadcast = (theta[:, None, :] * mx.ones((1, ratio, 1))).reshape(H, N // 2)

        angle_step = mx.einsum('blh, hn -> blhn', dt, theta_broadcast)
        if cache is not None:
            last_in_sig, h_state, prev_angle = cache
            angles = prev_angle + angle_step
            next_angle = angles
        else:
            angles = mx.cumsum(angle_step, axis=1)
            next_angle = angles[:, -1:]

        B_param_normed = self.norm_B(B_param.reshape(B_sz, L, G, N * R)).reshape(B_sz, L, G, N, R) + self.bias_B
        C_param_normed = self.norm_C(C_param.reshape(B_sz, L, G, N * R)).reshape(B_sz, L, G, N, R) + self.bias_C

        B_bcast = (B_param_normed[:, :, :, None, :, :] * mx.ones((1, 1, 1, ratio, 1, 1))).reshape(B_sz, L, H, N, R)
        C_bcast = (C_param_normed[:, :, :, None, :, :] * mx.ones((1, 1, 1, ratio, 1, 1))).reshape(B_sz, L, H, N, R)

        B_rotated = self.apply_rope(B_bcast, angles)
        C_rotated = self.apply_rope(C_bcast, angles)

        x = self.x_up_proj(x_prime).reshape(B_sz, L, H, P, R)

        input_signal = mx.einsum('blhnr, blhpr -> blhnp', B_rotated, x)
        lambda_view = (mx.sigmoid(lambda_param)[:, :, :, None] * mx.ones((1, 1, 1, ratio))).reshape(B_sz, L, H, 1, 1)
        dt_view = dt.reshape(B_sz, L, H, 1, 1)
        
        # 🌟 擷取 Forget Gate (Alpha)
        alpha_view = mx.exp(dt * A_broadcast).reshape(B_sz, L, H, 1, 1)

        if cache is not None:
            u_ssm = lambda_view * dt_view * input_signal + (1 - lambda_view) * dt_view * alpha_view * last_in_sig
            h_state = h_state * alpha_view[:, 0] + u_ssm[:, 0]
            y_stack = mx.einsum('bhnp, bhnr -> bhpr', h_state, C_rotated[:, 0])[:, None, ...]
            new_cache = (input_signal, h_state, next_angle)
        else:
            input_signal_prev = mx.concatenate([mx.zeros_like(input_signal[:, :1]), input_signal[:, :-1]], axis=1)
            u_ssm = lambda_view * dt_view * input_signal + (1 - lambda_view) * dt_view * alpha_view * input_signal_prev
            
            h_state = mx.zeros((B_sz, H, N, P), dtype=u.dtype)
            y_list = []
            for t in range(L):
                h_state = h_state * alpha_view[:, t] + u_ssm[:, t]
                y_t = mx.einsum('bhnp, bhnr -> bhpr', h_state, C_rotated[:, t])
                y_list.append(y_t)
            y_stack = mx.stack(y_list, axis=1)
            new_cache = (input_signal[:, -1:], h_state, next_angle)

        y = self.y_down_proj(y_stack.reshape(B_sz, L, H, P * R)).reshape(B_sz, L, H * P)
        y = y + x_prime.reshape(B_sz, L, H * P) * mx.repeat(self.D, P, axis=0)

        y = self.pre_gate_norm(y)
        y = y * nn.silu(z)

        # 🌟 回傳: 輸出、快取、以及 Forget Gate (平均值)
        return self.out_proj(y), new_cache, mx.mean(alpha_view)


# ==========================================
# 4. K-MoE Transformer FeedForward (不變)
# ==========================================
class KMoEFeedForward(nn.Module):
    def __init__(self, config: Mamba3Config):
        super().__init__()
        d1, d2 = get_factors(config.d_model)
        f1, f2 = get_factors(config.d_model * 4)
        self.up_proj = KroneckerMoE(d1, d2, f1, f2, config.kmoe_num_experts, config.kmoe_top_k)
        self.down_proj = KroneckerMoE(f1, f2, d1, d2, config.kmoe_num_experts, config.kmoe_top_k)

    def __call__(self, x):
        return self.down_proj(nn.gelu(self.up_proj(x)))


# ==========================================
# 5. Transformer Block 
# ==========================================
class TransformerBlock(nn.Module):
    def __init__(self, config: Mamba3Config):
        super().__init__()
        self.n_heads = config.d_model // 64
        self.d_head = 64
        self.qkv_proj = nn.Linear(config.d_model, 3 * config.d_model)
        self.out_proj = nn.Linear(config.d_model, config.d_model)
        self.norm_attn = nn.RMSNorm(config.d_model)
        self.norm_ffn = nn.RMSNorm(config.d_model)
        self.ffn = KMoEFeedForward(config) if config.use_kmoe else None
        
    def __call__(self, x, cache=None):
        B, L, D = x.shape
        h = self.norm_attn(x)
        
        qkv = self.qkv_proj(h)
        q, k, v = mx.split(qkv, 3, axis=-1)
        
        q = q.reshape(B, L, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        k = k.reshape(B, L, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        v = v.reshape(B, L, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        
        if cache is not None:
            k_cache, v_cache = cache
            k = mx.concatenate([k_cache, k], axis=2)
            v = mx.concatenate([v_cache, v], axis=2)
            
        new_cache = (k, v)
            
        scale = 1.0 / math.sqrt(self.d_head)
        scores = mx.matmul(q, k.transpose(0, 1, 3, 2)) * scale
        
        if cache is None and L > 1:
            mask = nn.MultiHeadAttention.create_additive_causal_mask(L, x.dtype)
            scores = scores + mask
            
        p = mx.softmax(scores, axis=-1)
        attn_out = mx.matmul(p, v).transpose(0, 2, 1, 3).reshape(B, L, D)
        
        x = x + self.out_proj(attn_out)
        if self.ffn is not None:
            x = x + self.ffn(self.norm_ffn(x))
            
        # 🌟 Transformer 沒有 Forget Gate，回傳 0.0
        return x, new_cache, mx.array(0.0)


# ==========================================
# 6. Hybrid Backbone 
# ==========================================
class LayerContainer(nn.Module):
    def __init__(self, l_type, norm, block):
        super().__init__()
        self.l_type = l_type
        if norm is not None:
            self.norm = norm
        self.block = block

class TrueHybridMamba(nn.Module):
    def __init__(self, config: Mamba3Config, mamba_ratio: int = 4):
        super().__init__()
        self.config = config
        self.mamba_ratio = mamba_ratio
        self.layers = []
        self.layer_scales = []
        
        for i in range(config.num_layers):
            for _ in range(mamba_ratio):
                self.layers.append(LayerContainer("mamba", nn.RMSNorm(config.d_model), Mamba3Block(config)))
                self.layer_scales.append(mx.ones((config.d_model,)) * 0.05)
            self.layers.append(LayerContainer("transformer", None, TransformerBlock(config)))
            self.layer_scales.append(mx.ones((config.d_model,)) * 0.01)

    def __call__(self, x, caches=None):
        if caches is None:
            caches = [None] * len(self.layers)
            
        new_caches = []
        magnitudes, update_ratios, forgets, cos_sims = [], [], [], []
        
        for i, layer_dict in enumerate(self.layers):
            scale = self.layer_scales[i]
            x_norm = mx.linalg.norm(x, axis=-1).mean()
            
            if layer_dict.l_type == "mamba":
                out, c, f = layer_dict.block(layer_dict.norm(x), cache=caches[i])
            else:
                out, c, f = layer_dict.block(x, cache=caches[i])
                
            layer_out = out * scale
            x_out = x + layer_out
            
            # A. 殘差更新率
            mag = mx.linalg.norm(layer_out, axis=-1).mean()
            ratio = mag / (x_norm + 1e-6)
            
            # D. 餘弦相似度 (x vs x_out)
            cos_sim = mx.mean(mx.sum(x * x_out, axis=-1) / (mx.linalg.norm(x, axis=-1) * mx.linalg.norm(x_out, axis=-1) + 1e-6))
            
            magnitudes.append(mag)
            update_ratios.append(ratio)
            forgets.append(f)
            cos_sims.append(cos_sim)
            
            x = x_out
            new_caches.append(c)
            
        return x, new_caches, mx.stack(magnitudes), mx.stack(update_ratios), mx.stack(forgets), mx.stack(cos_sims)
    
class Mamba3LanguageModel(nn.Module):
    def __init__(self, config: Mamba3Config):
        super().__init__()
        self.embed = nn.Embedding(config.vocab_size, config.d_model)
        self.backbone = TrueHybridMamba(config)
        self.norm = nn.RMSNorm(config.d_model)
        self.head = lambda x: mx.matmul(x, self.embed.weight.T)

    def __call__(self, input_ids, caches=None):
        x = self.embed(input_ids)
        x, new_caches, mags, ratios, forgets, cos_sims = self.backbone(x, caches) 
        x = self.norm(x)
        logits = self.head(x)
        return 30.0 * mx.tanh(logits / 30.0), new_caches, mags, ratios, forgets, cos_sims
    
# ==========================================
# 🌟 工具：字元自動換行與學術指標函數
# ==========================================
class LineWrapPrinter:
    def __init__(self, max_width=80):
        self.max_width = max_width
        self.curr_width = 0

    def write(self, text):
        for char in text:
            if char == '\n':
                self.curr_width = 0
                sys.stdout.write(char)
                continue
            
            sys.stdout.write(char)
            self.curr_width += 1
            
            if self.curr_width >= self.max_width:
                if char.isspace() or char in ".,!?。，！？":
                    sys.stdout.write('\n')
                    self.curr_width = 0
                elif self.curr_width >= self.max_width + 15: 
                    sys.stdout.write('\n')
                    self.curr_width = 0
        sys.stdout.flush()

def calculate_entropy_bits(logits_np):
    logits_np = logits_np - np.max(logits_np)
    probs = np.exp(logits_np) / np.sum(np.exp(logits_np))
    entropy = -np.sum(probs * np.log(probs + 1e-10))
    return entropy / np.log(2) 

def get_max_mamba_state_l2(caches):
    l2_max = 0.0
    for c in caches:
        if c is not None and len(c) == 3: 
            h_state = np.array(c[1])
            l2 = np.linalg.norm(h_state)
            if l2 > l2_max: l2_max = l2
    return l2_max

def format_expert_usage_heatmap(counts, num_bins=16):
    total = counts.sum()
    if total == 0:
        return ""

    max_usage_idx = np.argmax(counts)
    max_usage_pct = (counts[max_usage_idx] / total) * 100
    min_usage_pct = (counts.min() / total) * 100
    zero_count = np.sum(counts == 0)

    res = []
    res.append("\n" + "="*60)
    res.append("📊 [Expert Routing Heatmap & Statistics]")
    res.append("-" * 60)
    res.append(f"🎯 Total Routing Decisions : {total}")
    res.append(f"👑 Max Used Expert       : ID {max_usage_idx} ({max_usage_pct:.2f}%)")
    res.append(f"🧊 Min Used Expert       : {min_usage_pct:.2f}%")
    res.append(f"☠️  Dead Experts (0 usage) : {zero_count} / {len(counts)}")
    res.append("-" * 60)

    bin_size = len(counts) // num_bins if len(counts) >= num_bins else 1
    if len(counts) % num_bins == 0:
        binned = counts.reshape(num_bins, -1).sum(axis=1)
        max_bin = binned.max()
        for i, b_count in enumerate(binned):
            start = i * bin_size
            end = start + bin_size - 1
            pct = (b_count / total) * 100
            bar_len = int((b_count / max_bin) * 25) if max_bin > 0 else 0
            bar = "█" * bar_len
            res.append(f"Exp {start:03d}-{end:03d} | {bar:<25} | {pct:>5.2f}%")
    else:
        top_indices = np.argsort(counts)[::-1][:10]
        max_val = counts[top_indices[0]]
        for idx in top_indices:
            pct = (counts[idx] / total) * 100
            bar_len = int((counts[idx] / max_val) * 25) if max_val > 0 else 0
            bar = "█" * bar_len
            res.append(f"Expert {idx:03d} | {bar:<25} | {pct:>5.2f}%")

    res.append("=" * 60 + "\n")
    return "\n".join(res)



# ==========================================
# 🌟 10. [Layer-wise Diagnostics] 列印每一層的詳細數值
# ==========================================
def print_layer_diagnostics(model_backbone, magnitudes_list, current_caches):
    print("\n" + "="*70)
    print("🔍 [Layer-wise Diagnostics] 每層數值分析 (抓取數值異常層)")
    print("-" * 70)
    print(f"{'Layer':<8} | {'Type':<15} | {'Avg Output Mag (Scaled)':<25} | {'Final State L2 (Raw)':<20}")
    print("-" * 70)
    
    # 計算所有 Token 生成過程中，每一層的平均 Magnitude
    heatmap_data = np.stack(magnitudes_list, axis=1)
    layer_avg_mags = np.mean(heatmap_data, axis=1)
    
    for i, layer in enumerate(model_backbone.layers):
        l_type = "Transformer" if layer.l_type == "transformer" else "Mamba"
        avg_mag = layer_avg_mags[i]
        
        # 取得 Cache 中真實的 Mamba State L2 Norm
        state_l2_str = "N/A (No State)"
        if current_caches[i] is not None and len(current_caches[i]) == 3:
            # current_caches[i] 裡面有 (input_signal, h_state, next_angle)
            h_state = np.array(current_caches[i][1])
            state_l2 = np.linalg.norm(h_state)
            state_l2_str = f"{state_l2:.4f}"
            
            # 如果 L2 大於 100，標示警告 ⚠️
            if state_l2 > 100.0:
                state_l2_str += " ⚠️ EXPLODING"
            
        print(f"L{i:<7} | {l_type:<15} | {avg_mag:<25.4f} | {state_l2_str:<20}")
    print("=" * 70 + "\n")



In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", type=str, default="The capital of France is")
    parser.add_argument("--checkpoint", type=str, default="/Users/hungwei/Downloads/mamba3_colab_checkpoint (1).pt")
    parser.add_argument("--tokenizer", type=str, default="/Users/hungwei/Desktop/Proj/Mamba3-XR/data/spm_tokenizer.model")
    parser.add_argument("--max_tokens", type=int, default=512)
    parser.add_argument("--temp", type=float, default=0.5)     
    parser.add_argument("--top_k", type=int, default=40)       
    parser.add_argument("--top_p", type=float, default=0.95)
    parser.add_argument("--min_p", type=float, default=0.05)   
    parser.add_argument("--rep_pen", type=float, default=1.2)  
    parser.add_argument("--pres_pen", type=float, default=0.0)
    parser.add_argument("--freq_pen", type=float, default=0.15) 
    
    parser.add_argument("--dtype", type=str, default="bfloat16", 
                        choices=["float32", "float16", "bfloat16", "int8", "int4"], 
                        help="設定模型推理時的精度與量化位元數")

    args = parser.parse_args(args=[])
    mx.set_default_device(mx.gpu)
    
    args.temp = 0.35
    args.top_k = 25
    args.top_p = 0.9
    args.rep_pen = 1.15
    args.freq_pen = 0.12

    if not os.path.exists(args.tokenizer):
        print(f"⚠️ Tokenizer 檔案不存在: {args.tokenizer}")
        sys.exit(1)

    tokenizer = spm.SentencePieceProcessor(model_file=args.tokenizer)
    NUM_EXPERTS = 256
    
    config = Mamba3Config(
        d_model=768, d_state=64, expand=4, num_layers=5, # 依照你圖片層數示意，可以調整這裡
        vocab_size=tokenizer.vocab_size(), 
        use_kmoe=True, kmoe_num_experts=NUM_EXPERTS, 
        mimo_rank=4, kmoe_top_k=4
    )
    model = Mamba3LanguageModel(config)
    
    if os.path.exists(args.checkpoint):
        print(f"📥 正在讀取 PyTorch Checkpoint ({args.checkpoint}) 並強制注入 MLX 模型...")
        ckpt = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
        
        target_dtype = mx.float16 if args.dtype in ["int8", "int4"] else getattr(mx, args.dtype) 
        
        success_count = 0
        failed_keys = []
        
        for k, v_pt in ckpt["model"].items():
            new_k = k[10:] if k.startswith("_orig_mod.") else (k[7:] if k.startswith("module.") else k)
            new_k = new_k.replace(".attn.in_proj_weight", ".qkv_proj.weight")
            new_k = new_k.replace(".attn.in_proj_bias", ".qkv_proj.bias")
            new_k = new_k.replace(".attn.out_proj.weight", ".out_proj.weight")
            new_k = new_k.replace(".attn.out_proj.bias", ".out_proj.bias")
            new_k = new_k.replace(".o_proj.weight", ".out_proj.weight")
            new_k = new_k.replace(".o_proj.bias", ".out_proj.bias")
            
            v_pt = v_pt.to(torch.float32)
            v_mx = mx.array(v_pt.numpy(), dtype=target_dtype)
            
            parts = new_k.split('.')
            obj = model
            try:
                for part in parts[:-1]:
                    if part.isdigit() and isinstance(obj, list):
                        obj = obj[int(part)]
                    else:
                        obj = getattr(obj, part)
                
                last_part = parts[-1]
                if last_part.isdigit() and isinstance(obj, list):
                    obj[int(last_part)] = v_mx
                else:
                    setattr(obj, last_part, v_mx)
                success_count += 1
            except Exception as e:
                failed_keys.append(f"原始 Key: '{k}' -> 對應 Key: '{new_k}' | 錯誤訊息: {str(e)}")

        del ckpt; gc.collect()
        
        if failed_keys:
            error_msg = f"\n❌ 權重注入失敗！共有 {len(failed_keys)} 個參數無法成功載入。\n"
            error_msg += "請檢查您的模型結構（Mamba3Config）是否與 Checkpoint 吻合。詳細錯誤清單如下：\n"
            error_msg += "-" * 60 + "\n"
            error_msg += "\n".join(failed_keys)
            error_msg += "\n" + "-" * 60
            raise RuntimeError(error_msg)
            
        mx.eval(model.parameters()) 
        print(f"✅ 模型權重強制注入完成！基礎精度設定為: {target_dtype} (成功: {success_count})")

    total_params, active_params = 0, 0
    total_param_bytes, active_param_bytes = 0, 0
    model_dtype_str = args.dtype

    for path, v in tree_flatten(model.parameters()):
        total_params += v.size
        total_param_bytes += v.nbytes
        if "A_experts" in path or "B_experts" in path:
            num_experts = v.shape[0]
            top_k = config.kmoe_top_k
            active_params += int(v.size * (top_k / num_experts))
            active_param_bytes += int(v.nbytes * (top_k / num_experts))
        else:
            active_params += v.size
            active_param_bytes += v.nbytes

    def sample_pt(logits_np, generated_ids):
        logits = torch.from_numpy(logits_np)
        if len(generated_ids) > 0:
            unique_tokens = set(generated_ids)
            for token_id in unique_tokens:
                count = generated_ids.count(token_id)
                score = logits[0, token_id].item()
                if args.rep_pen != 1.0: 
                    score = score * args.rep_pen if score < 0 else score / args.rep_pen
                if args.pres_pen > 0.0: 
                    score = score - args.pres_pen
                if args.freq_pen > 0.0:
                    score = score - (args.freq_pen * count)
                logits[0, token_id] = score
                
        if args.temp != 1.0: logits = logits / args.temp
        
        if args.min_p > 0.0:
            probs = F.softmax(logits, dim=-1)
            max_prob = torch.max(probs, dim=-1, keepdim=True).values
            scaled_min_p = max_prob * args.min_p
            logits[probs < scaled_min_p] = -float("inf")
            
        if args.top_k is not None and args.top_k > 0:
            v, idx = torch.topk(logits, args.top_k)
            logits_filtered = torch.full_like(logits, float("-inf"))
            logits_filtered.scatter_(1, idx, v)
            logits = logits_filtered
            
        if args.top_p is not None and args.top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            probs = F.softmax(sorted_logits, dim=-1)
            cumulative = torch.cumsum(probs, dim=-1)
            mask = cumulative > args.top_p
            mask[..., 1:] = mask[..., :-1].clone()
            mask[..., 0] = False
            sorted_logits[mask] = -float("inf")
            logits = torch.full_like(logits, -float("inf"))
            logits.scatter_(1, sorted_idx, sorted_logits)
            
        return torch.multinomial(F.softmax(logits, dim=-1), 1).item()

    expert_counts = np.zeros(NUM_EXPERTS, dtype=int)
    total_entropy = 0.0
    total_confidence = 0.0
    confidence_steps = 0
    max_state_l2 = 0.0

    # 🌟 1. 建立存放 Magnitude 的 List
    all_magnitudes = []
    
    expert_counts = np.zeros(NUM_EXPERTS, dtype=int)
    total_entropy = 0.0
    total_confidence = 0.0
    confidence_steps = 0
    max_state_l2 = 0.0

    # 🌟 1. 建立存放 4 大指標的 List
    all_magnitudes, all_update_ratios, all_forgets, all_cos_sims, all_moe_counts = [], [], [], [], []

    # 🌟 2. 更新靜態圖 decode_step 接收新指標
    @mx.compile
    def decode_step(x_in, current_caches):
        x_emb = model.embed(x_in)
        out_logits, new_caches, mags, ratios, forgets, cos_sims = model.backbone(x_emb, caches=current_caches)
        out_logits = model.norm(out_logits)
        out_logits = model.head(out_logits)
        out_logits = 30.0 * mx.tanh(out_logits / 30.0)
        
        step_indices_list = [v for k, v in tree_flatten(model) if "step_indices" in k]
        step_conf_list = [v for k, v in tree_flatten(model) if "step_confidence" in k]
        
        return out_logits[:, -1, :], new_caches, step_indices_list, step_conf_list, mags, ratios, forgets, cos_sims

    printer = LineWrapPrinter(max_width=80) 
    print("\n🚀 開始執行文本生成...", end="")
    prompt_ids = tokenizer.encode(args.prompt)
    history_ids = prompt_ids.copy()
    generated_ids = prompt_ids.copy()
    printed_text = tokenizer.decode(generated_ids)
    printer.write(printed_text)

    # ==========================================
    # 第一步：Prefill 階段
    # ==========================================
    t_prefill_start = time.time()
    
    x_prefill = mx.array([prompt_ids])
    logits, caches, step_indices_list, step_conf_list, mags, ratios, forgets, cos_sims = decode_step(x_prefill, None)
    
    # 將新指標全部丟入 mx.eval
    mx.eval(logits, *step_indices_list, *step_conf_list, mags, ratios, forgets, cos_sims, *[v for _, v in tree_flatten(caches)])
    
    # 儲存指標
    all_magnitudes.append(np.array(mags))
    all_update_ratios.append(np.array(ratios))
    all_forgets.append(np.array(forgets))
    all_cos_sims.append(np.array(cos_sims))
    
    # 儲存 MoE Experts
    step_expert_counts = np.zeros(NUM_EXPERTS, dtype=int)
    for indices_array in step_indices_list:
        indices_np = np.array(indices_array).flatten()
        for idx in indices_np:
            expert_counts[idx] += 1
            step_expert_counts[idx] += 1
    all_moe_counts.append(step_expert_counts)
            
    for conf_array in step_conf_list:
        total_confidence += float(np.array(conf_array.astype(mx.float32)).mean())
        confidence_steps += 1
        
    logits_step_np = np.array(logits)
    total_entropy += calculate_entropy_bits(logits_step_np[0])
    
    current_l2 = get_max_mamba_state_l2(caches)
    if current_l2 > max_state_l2: max_state_l2 = current_l2
    
    t_prefill_end = time.time()
    ttft = t_prefill_end - t_prefill_start
    prefill_tps = len(prompt_ids) / ttft if ttft > 0 else 0.0
    
    next_token = sample_pt(logits_step_np, history_ids)
    history_ids.append(next_token)
    generated_ids.append(next_token) 
    current_text = tokenizer.decode(generated_ids)
    printer.write(current_text[len(printed_text):])
    printed_text = current_text

    # ==========================================
    # 第二步：Decode 階段 
    # ==========================================
    _ = decode_step(mx.array([[next_token]]), caches)
    
    t_decode_start = time.time()
    for _ in range(args.max_tokens - 1):
        x_in = mx.array([[next_token]]) 
        logits, caches, step_indices_list, step_conf_list, mags, ratios, forgets, cos_sims = decode_step(x_in, caches)
        
        mx.eval(logits, *step_indices_list, *step_conf_list, mags, ratios, forgets, cos_sims, *[v for _, v in tree_flatten(caches)])
        
        all_magnitudes.append(np.array(mags))
        all_update_ratios.append(np.array(ratios))
        all_forgets.append(np.array(forgets))
        all_cos_sims.append(np.array(cos_sims))
        
        step_expert_counts = np.zeros(NUM_EXPERTS, dtype=int)
        for indices_array in step_indices_list:
            indices_np = np.array(indices_array).flatten()
            for idx in indices_np:
                expert_counts[idx] += 1
                step_expert_counts[idx] += 1
        all_moe_counts.append(step_expert_counts)
                
        for conf_array in step_conf_list:
            total_confidence += float(np.array(conf_array.astype(mx.float32)).mean())
            confidence_steps += 1
            
        logits_step_np = np.array(logits)
        total_entropy += calculate_entropy_bits(logits_step_np[0])
        next_token = sample_pt(logits_step_np, history_ids)
        
        current_l2 = get_max_mamba_state_l2(caches)
        if current_l2 > max_state_l2: max_state_l2 = current_l2
        
        history_ids.append(next_token)
        generated_ids.append(next_token)
        
        current_text = tokenizer.decode(generated_ids)
        printer.write(current_text[len(printed_text):])
        printed_text = current_text
        
    t_decode_end = time.time()

    # 計算效能與學術指標
    decode_time = t_decode_end - t_decode_start
    tpot = decode_time / (args.max_tokens - 1)
    decode_tps = (args.max_tokens - 1) / decode_time if decode_time > 0 else 0.0
    prefill_flops = 2.0 * active_params * len(prompt_ids)
    prefill_tflops = (prefill_flops / ttft) / 1e12 if ttft > 0 else 0.0
    decode_flops_per_token = 2.0 * active_params
    decode_tflops = (decode_flops_per_token * decode_tps) / 1e12
    avg_confidence = (total_confidence / confidence_steps * 100) if confidence_steps > 0 else 0.0
    avg_entropy = total_entropy / args.max_tokens
    
    print(f"\n✅ 文本生成完畢！ TPS: {decode_tps:.2f} | Router Conf: {avg_confidence:.2f}% | Entropy: {avg_entropy:.2f}")

    # ==========================================
    # 🌟 3. 繪製 4 種高階互動式熱力圖 (Plotly)
    # ==========================================
    def plot_interactive_diagnostics(all_ratios, all_forgets, all_cos_sims, all_moe_counts, mamba_ratio=4):
        actual_num_layers = len(all_ratios[0])
        yticks_labels = [f"L{i} (T)" if (i+1) % (mamba_ratio+1) == 0 else f"L{i} (M)" for i in range(actual_num_layers)]

        # 建立 2x2 的網格圖表
        fig = make_subplots(rows=2, cols=2,
                            subplot_titles=("A. 殘差更新率 (Residual Update Ratio)",
                                            "B. Mamba 遺忘閘門 (Forget Gate α)",
                                            "C. MoE 專家負載平衡 (Routing Matrix)",
                                            "D. 餘弦相似度演化 (Representation Flow)"),
                            vertical_spacing=0.15,
                            horizontal_spacing=0.1)

        # A. Update Ratio
        z_ratios = np.stack(all_ratios, axis=1)
        fig.add_trace(go.Heatmap(z=z_ratios, y=yticks_labels, colorscale="Viridis", 
                                 colorbar=dict(title="Ratio", x=0.45, y=0.78, len=0.4)), row=1, col=1)

        # B. Forget Gate (將 Transformer 的 0.0 設為灰色或深色)
        z_forgets = np.stack(all_forgets, axis=1)
        fig.add_trace(go.Heatmap(z=z_forgets, y=yticks_labels, colorscale="Magma", 
                                 colorbar=dict(title="α Value", x=1.0, y=0.78, len=0.4)), row=1, col=2)

        # C. MoE Routing Heatmap
        z_moe = np.stack(all_moe_counts, axis=1)
        fig.add_trace(go.Heatmap(z=z_moe, colorscale="Plasma", 
                                 colorbar=dict(title="Usage Count", x=0.45, y=0.22, len=0.4)), row=2, col=1)
        fig.update_yaxes(title_text="Expert ID (0-255)", row=2, col=1)

        # D. Cosine Similarity
        z_cos = np.stack(all_cos_sims, axis=1)
        fig.add_trace(go.Heatmap(z=z_cos, y=yticks_labels, colorscale="Cividis", 
                                 colorbar=dict(title="Cos Sim", x=1.0, y=0.22, len=0.4)), row=2, col=2)

        # 調整版面與標籤
        fig.update_xaxes(title_text="Generated Tokens (Time step)")
        fig.update_layout(height=1000, width=1500, title_text="Hybrid Mamba+Transformer 深度互動式診斷圖表")

        # 儲存為可以滑鼠互動的 HTML
        html_file = "interactive_diagnostics.html"
        fig.write_html(html_file)
        print(f"\n📊 互動式熱力圖已儲存為 {html_file} ！ (請在資料夾中點開使用瀏覽器查看，可懸停查看數值)")
        
        try:
            fig.show() # 在 Jupyter 中直接顯示互動圖表
        except:
            pass

    # 執行繪圖
    plot_interactive_diagnostics(all_update_ratios, all_forgets, all_cos_sims, all_moe_counts)

📥 正在讀取 PyTorch Checkpoint (/Users/hungwei/Downloads/mamba3_colab_checkpoint (1).pt) 並強制注入 MLX 模型...
✅ 模型權重強制注入完成！基礎精度設定為: mlx.core.bfloat16 (成功: 578)

🚀 開始執行文本生成...The capital of France is a the world. It also was not an important part in this 
way, and it’s that you have been able to be working on your own family or their 
life-related health. This can help us to get more than one thing about our people 
who are trying to do with them as well. If we know how they don’t need for any time 
when there will make sure what I think? How does these things like: - You may want 
to see if he would take some other ways to keep up out by using all types of information 
(or) - What could me learn from my home! But because she's going into which her 
first day has had no data – but then just being done at least another year; so many 
years ago, “I really believe” were still looking forward through something else.
” When someone knows why everyone didn’t work before his wife came back again — 
even

In [14]:
# 设置为推理模式
!export MLX_TRAINING=0

In [15]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", type=str, default="The capital of France is")
    parser.add_argument("--checkpoint", type=str, default="/Users/hungwei/Downloads/3400.pt")
    parser.add_argument("--tokenizer", type=str, default="/Users/hungwei/Desktop/Proj/Mamba3-XR/data/spm_tokenizer.model")
    parser.add_argument("--max_tokens", type=int, default=128)
    
    # 🌟 收緊的 Hyperparameters 預設值 (避免模型胡言亂語)
    parser.add_argument("--temp", type=float, default=0.5)     # 極低溫，接近 Greedy Decode
    parser.add_argument("--top_k", type=int, default=40)       
    parser.add_argument("--top_p", type=float, default=0.95)
    parser.add_argument("--min_p", type=float, default=0.05)   # Min-P 採樣
    parser.add_argument("--rep_pen", type=float, default=1.2)  # 關閉懲罰，測試純粹生成能力
    parser.add_argument("--pres_pen", type=float, default=0.0)
    parser.add_argument("--freq_pen", type=float, default=0.15) 
    
    parser.add_argument("--dtype", type=str, default="bfloat16", 
                        choices=["float32", "float16", "bfloat16", "int8", "int4"], 
                        help="設定模型推理時的精度與量化位元數")

    args = parser.parse_args(args=[])
    mx.set_default_device(mx.gpu)

    if not os.path.exists(args.tokenizer):
        print(f"⚠️ Tokenizer 檔案不存在: {args.tokenizer}")
        sys.exit(1)

    tokenizer = spm.SentencePieceProcessor(model_file=args.tokenizer)
    NUM_EXPERTS = 256
    
    config = Mamba3Config(
        d_model=768, d_state=64, expand=4, num_layers=5, 
        vocab_size=tokenizer.vocab_size(), 
        use_kmoe=True, kmoe_num_experts=NUM_EXPERTS, 
        mimo_rank=4, kmoe_top_k=4
    )
    model = Mamba3LanguageModel(config)
    
    if os.path.exists(args.checkpoint):
        print(f"📥 正在讀取 PyTorch Checkpoint ({args.checkpoint}) 並強制注入 MLX 模型...")
        ckpt = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
        
        target_dtype = mx.float16 if args.dtype in ["int8", "int4"] else getattr(mx, args.dtype) 
        
        success_count, fail_count = 0, 0
        for k, v_pt in ckpt["model"].items():
            new_k = k[10:] if k.startswith("_orig_mod.") else (k[7:] if k.startswith("module.") else k)
            new_k = new_k.replace(".attn.in_proj_weight", ".qkv_proj.weight")
            new_k = new_k.replace(".attn.in_proj_bias", ".qkv_proj.bias")
            new_k = new_k.replace(".attn.out_proj.weight", ".out_proj.weight")
            new_k = new_k.replace(".attn.out_proj.bias", ".out_proj.bias")
            
            v_pt = v_pt.to(torch.float32)
            v_mx = mx.array(v_pt.numpy(), dtype=target_dtype)
            
            parts = new_k.split('.')
            obj = model
            try:
                for part in parts[:-1]:
                    if part.isdigit() and isinstance(obj, list):
                        obj = obj[int(part)]
                    else:
                        obj = getattr(obj, part)
                
                last_part = parts[-1]
                if last_part.isdigit() and isinstance(obj, list):
                    obj[int(last_part)] = v_mx
                else:
                    setattr(obj, last_part, v_mx)
                success_count += 1
            except Exception as e:
                fail_count += 1

        del ckpt; gc.collect()
        mx.eval(model.parameters()) 
        print(f"✅ 模型權重強制注入完成！基礎精度設定為: {target_dtype} (成功: {success_count}, 失敗: {fail_count})")
        
        if args.dtype in ["int8", "int4"]:
            print(f"🔄 正在啟動權重量化轉換 ({args.dtype})...")
            bits = 8 if args.dtype == "int8" else 4
            nn.quantize(model.backbone, group_size=64, bits=bits)
            mx.eval(model.parameters())
            print(f"✅ 模型量化為 {bits}-bit 完成！")
            
    else:
        print(f"⚠️ 找不到 Checkpoint 檔案，將使用隨機初始化的權重。")

    # 參數統計
    total_params, active_params = 0, 0
    total_param_bytes, active_param_bytes = 0, 0
    model_dtype_str = args.dtype

    for path, v in tree_flatten(model.parameters()):
        total_params += v.size
        total_param_bytes += v.nbytes
        if "A_experts" in path or "B_experts" in path:
            num_experts = v.shape[0]
            top_k = config.kmoe_top_k
            active_params += int(v.size * (top_k / num_experts))
            active_param_bytes += int(v.nbytes * (top_k / num_experts))
        else:
            active_params += v.size
            active_param_bytes += v.nbytes

    def sample_pt(logits_np, generated_ids):
        logits = torch.from_numpy(logits_np)
        if len(generated_ids) > 0:
            unique_tokens = set(generated_ids)
            for token_id in unique_tokens:
                count = generated_ids.count(token_id)
                score = logits[0, token_id].item()
                if args.rep_pen != 1.0: 
                    score = score * args.rep_pen if score < 0 else score / args.rep_pen
                if args.pres_pen > 0.0: 
                    score = score - args.pres_pen
                if args.freq_pen > 0.0:
                    score = score - (args.freq_pen * count)
                logits[0, token_id] = score
                
        if args.temp != 1.0: logits = logits / args.temp
        
        if args.min_p > 0.0:
            probs = F.softmax(logits, dim=-1)
            max_prob = torch.max(probs, dim=-1, keepdim=True).values
            scaled_min_p = max_prob * args.min_p
            logits[probs < scaled_min_p] = -float("inf")
            
        if args.top_k is not None and args.top_k > 0:
            v, idx = torch.topk(logits, args.top_k)
            logits_filtered = torch.full_like(logits, float("-inf"))
            logits_filtered.scatter_(1, idx, v)
            logits = logits_filtered
            
        if args.top_p is not None and args.top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            probs = F.softmax(sorted_logits, dim=-1)
            cumulative = torch.cumsum(probs, dim=-1)
            mask = cumulative > args.top_p
            mask[..., 1:] = mask[..., :-1].clone()
            mask[..., 0] = False
            sorted_logits[mask] = -float("inf")
            logits = torch.full_like(logits, -float("inf"))
            logits.scatter_(1, sorted_idx, sorted_logits)
            
        return torch.multinomial(F.softmax(logits, dim=-1), 1).item()

    # 🌟 XLA 風格：將 Decode 步驟包裝為純函數並進行靜態編譯
    @mx.compile
    def decode_step(x_in, current_caches):
        x_emb = model.embed(x_in)
        out_logits, new_caches = model.backbone(x_emb, caches=current_caches)
        out_logits = model.norm(out_logits)
        out_logits = model.head(out_logits)
        out_logits = 30.0 * mx.tanh(out_logits / 30.0)
        return out_logits[:, -1, :], new_caches

    printer = LineWrapPrinter(max_width=80) 

    print("\n🚀 開始執行文本生成...", end="")
    prompt_ids = tokenizer.encode(args.prompt)
    history_ids = prompt_ids.copy()
    generated_ids = prompt_ids.copy()
    printed_text = tokenizer.decode(generated_ids)
    printer.write(printed_text)

    # ==========================================
    # 第一步：Prefill 階段
    # ==========================================
    t_prefill_start = time.time()
    
    x_prefill = mx.array([prompt_ids])
    logits, caches = decode_step(x_prefill, None)
    mx.eval(logits, *[v for _, v in tree_flatten(caches)])
    
    t_prefill_end = time.time()
    ttft = t_prefill_end - t_prefill_start
    prefill_tps = len(prompt_ids) / ttft if ttft > 0 else 0.0
    
    logits_step = np.array(logits)
    next_token = sample_pt(logits_step, history_ids)
    history_ids.append(next_token)
    generated_ids.append(next_token) 
    
    current_text = tokenizer.decode(generated_ids)
    printer.write(current_text[len(printed_text):])
    printed_text = current_text

    # ==========================================
    # 第二步：Decode 階段 (XLA/JIT Compiled)
    # ==========================================
    # 預熱編譯器 (可選，讓第一次建圖的時間不計入 TPS)
    _ = decode_step(mx.array([[next_token]]), caches)
    
    t_decode_start = time.time()
    for _ in range(args.max_tokens - 1):
        x_in = mx.array([[next_token]]) 
        
        # ⚡️ 呼叫編譯後的靜態圖
        logits, caches = decode_step(x_in, caches)
        
        # ⚡️ 強制 GPU 計算，但不拉回 CPU 進行額外統計 (消除 Side Effects)
        mx.eval(logits, *[v for _, v in tree_flatten(caches)])
        
        # 唯一的 Sync：將 logits 轉回 CPU 進行採樣
        logits_step_np = np.array(logits)
        next_token = sample_pt(logits_step_np, history_ids)
        
        history_ids.append(next_token)
        generated_ids.append(next_token)
        
        current_text = tokenizer.decode(generated_ids)
        printer.write(current_text[len(printed_text):])
        printed_text = current_text
        
    t_decode_end = time.time()

    cache_bytes = 0
    for c in caches:
        if c is not None:
            for item in c:
                if isinstance(item, mx.array):
                    cache_bytes += item.nbytes

    metal_mem_info = ""
    try:
        import mlx.metal as mx_metal
        metal_mem_info = f"""
💻 [Hardware] Apple Silicon Metal GPU Memory
------------------------------------------------------------
🚀 Peak Memory        : {mx_metal.get_peak_memory() / (1024**2):.2f} MB
🟢 Active Memory      : {mx_metal.get_active_memory() / (1024**2):.2f} MB
🗄️ Metal API Cache    : {mx_metal.get_cache_memory() / (1024**2):.2f} MB"""
    except Exception:
        pass

    # ==========================================
    # 計算 TPS 與 TFLOPS
    # ==========================================
    decode_time = t_decode_end - t_decode_start
    tpot = decode_time / (args.max_tokens - 1)
    decode_tps = (args.max_tokens - 1) / decode_time if decode_time > 0 else 0.0

    # 1 個 Token 的 FLOPs 約為 2 * 活躍參數量
    prefill_flops = 2.0 * active_params * len(prompt_ids)
    prefill_tflops = (prefill_flops / ttft) / 1e12 if ttft > 0 else 0.0

    decode_flops_per_token = 2.0 * active_params
    decode_tflops = (decode_flops_per_token * decode_tps) / 1e12

    metrics_text = f"""
\n============================================================
📈 [Academic Metrics] Architecture & Efficiency
------------------------------------------------------------
⚙️ Precision          : {model_dtype_str}
📦 Total Params       : {total_params / 1e6:.2f} M
⚡ Active Params      : {active_params / 1e6:.2f} M
🎯 % Active           : {(active_params / total_params) * 100:.2f} %
------------------------------------------------------------
💾 [Memory Footprint] Theoretical Context
------------------------------------------------------------
📚 Context Cache Mem  : {cache_bytes / (1024**2):.2f} MB (KV & Mamba States){metal_mem_info}
------------------------------------------------------------
⚡️ [Performance] JIT Compiled Compute Metrics
------------------------------------------------------------
⏱️  Time To First Token (TTFT) : {ttft:.3f} s
⏱️  Time Per Output Token (TPOT): {tpot:.3f} s/token
------------------------------------------------------------
📊  Prefill TPS               : {prefill_tps:.2f} tokens/s
🚀  Prefill Compute           : {prefill_tflops:.4f} TFLOPS
------------------------------------------------------------
📊  Decode TPS                : {decode_tps:.2f} tokens/s
🚀  Decode Compute            : {decode_tflops:.4f} TFLOPS
============================================================
"""
    print(metrics_text)

⚠️ 找不到 Checkpoint 檔案，將使用隨機初始化的權重。

🚀 開始執行文本生成...The capital of France is

ValueError: too many values to unpack (expected 2, got 6)